# Advanced U-Net: TensorFlow on DRISTI

This notebook demonstrates the modular project workflow for optic disc and optic cup segmentation.
All reusable implementation code lives in `src/`.

Run cells from the project root so imports resolve correctly.

## Setup

Install dependencies first:

```bash
pip install -r requirements.txt
```

Place the extracted DRISTI dataset inside `dataset/` before running the cells below.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
  PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
  sys.path.insert(0, str(PROJECT_ROOT))

from src.dataset import prepare_datasets
from src.model import build_unet
from src.predict import (
  evaluate_each_test_image,
  evaluate_model,
  load_trained_model,
  predict_single_image,
  show_disc_and_cup_samples,
  show_predictions,
)
from src.train import compile_model, plot_training_history, train_model
from src.utils import Config, configure_gpu, set_random_seed

config = Config()
config.ensure_output_dir()

set_random_seed(config.seed)
configure_gpu()

print("DATA_ROOT:", config.data_root)
print("OUTPUT_DIR:", config.output_dir)
print("Targets:", config.target_display_names)

## Prepare Datasets

Discover image-mask triples, perform a stratified train/validation split, and build TensorFlow datasets.

In [ ]:
data = prepare_datasets(config)

train_ds = data["train_ds"]
val_ds = data["val_ds"]
test_ds = data["test_ds"]
test_pairs = data["test_pairs"]

display(data["trainval_df"].head())
print("Training label folders:")
print(data["trainval_df"]["label_folder"].value_counts())

## Visualize Training Samples

In [ ]:
show_disc_and_cup_samples(train_ds, count=4, title="Training Samples: Optic Disc and Cup Masks")
show_disc_and_cup_samples(val_ds, count=4, title="Validation Samples: Optic Disc and Cup Masks")

## Build and Inspect the Model

In [ ]:
model = build_unet(config)
compile_model(model, config)
model.summary()

## Train

For a quick classroom dry run, temporarily set `config.epochs = 3` before running the next cell.

In [ ]:
model, history = train_model(train_ds, val_ds, config)
plot_training_history(history)

## Evaluate and Visualize Predictions

In [ ]:
best_model = load_trained_model(config)
evaluate_model(best_model, val_ds, test_ds)

show_predictions(best_model, val_ds, count=4, threshold=0.5)
show_predictions(best_model, test_ds, count=4, threshold=0.5)

## Per-Image Test Metrics

In [ ]:
test_metrics_df = evaluate_each_test_image(best_model, test_pairs, config, threshold=0.5)
test_metrics_df.to_csv(config.test_metrics_csv, index=False)

print("Saved per-image metrics to:", config.test_metrics_csv)
display(test_metrics_df.head(10))
display(test_metrics_df.describe(numeric_only=True))

## Single-Image Inference

In [ ]:
example_image_path = test_pairs[0][0]
prediction_outputs = predict_single_image(best_model, example_image_path, config, threshold=0.5)